# Fine-tune XLM-RoBERTa on Vertex AI Workbench (CPU)

Run this notebook **on the gated Workbench instance**, not locally and
not on a Dataflow worker.

1. `n2-standard-16` (CPU) in `europe-central2-b` — no guest accelerator
2. Dataset cache: `gs://co-tf-artifacts-dev/nlp/datasets/`
3. MLflow: `gs://co-tf-artifacts-dev/nlp/mlruns` (experiment `xlmr-sentiment`)
4. **Stop the instance when the run finishes** — idle shutdown is off by
   default; a running VM still bills.

Model: `xlm-roberta-base`, 3-class (`neg`/`neu`/`pos`), `MAX_LEN=128`.
Recipe: `docs/phase-1-nlp-multilingual-v2.md`.

In [ ]:
import os
from pathlib import Path

# Repo root on the Workbench VM. Adjust if you cloned elsewhere.
REPO = Path.home() / "pop-vibe-check"
os.chdir(REPO)
%pip install -q -r nlp/training/requirements.txt

## Cache datasets from GCS

Hugging Face `cache_dir` is local. Rsync the shared prefix so a restarted
Workbench does not hit the Hub again. First session: download via the
loaders, then `gsutil -m rsync -r /home/jupyter/hf-datasets gs://co-tf-artifacts-dev/nlp/datasets/`.

In [ ]:
from nlp.training.loaders import DEFAULT_GCS_DATASETS_URI, cache_dir_from_gcs

CACHE = cache_dir_from_gcs(DEFAULT_GCS_DATASETS_URI, Path.home() / "hf-datasets")
print("cache", CACHE)

## Train

`train.py` loads the v2 mix (clapAI subsample, multilingual tweets,
downsampled tweet_eval / GoEmotions, oversampled gold), fine-tunes
XLM-RoBERTa, logs to MLflow. Pass `--own-domain` to the gold JSONL;
`split=holdout` rows are skipped.

In [ ]:
from nlp.training.train import main

main([
    "--cache-dir", str(CACHE),
    "--output-dir", str(Path.home() / "models" / "xlmr-sent"),
    "--epochs", "3",
    "--batch-size", "8",
    "--lr", "2e-5",
])

## After training

1. Copy the export to GCS:
   `gsutil -m cp -r ~/models/xlmr-sent gs://co-tf-artifacts-dev/nlp/models/`
2. `python -m nlp.endpoint.register upload --model-dir gs://… --display-name xlmr-sent`
3. Enable the Endpoint in Terraform, then `register deploy` (CPU:
   `n1-standard-8`, no accelerator).
4. **Stop this Workbench instance** (`gcloud workbench instances stop …` or
   `desired_state=STOPPED`). Do not leave the VM running overnight.

See `nlp/README.md` and `terraform/modules/vertex_nlp/README.md`.